In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os


## Seleccionar Directorio
***

In [2]:
#Seleccionamos el archivo que queremos trabajar

directorio_entrada  =          './Data/corrected_sections/A01'
directorio_salida   = './Data/corrected_sections_filtrado/A01'


## Ajustar filtro Hanning 
***

In [3]:
# Crear un kernel Hanning
kernel_width_dbar = 40  # Ancho del kernel en dbar
spatial_resolution = 1  # Definir spatial_resolution en dbar
kernel_size_points = int(kernel_width_dbar * spatial_resolution)
kernel = np.hanning(kernel_size_points)
kernel = kernel / kernel.sum()  # Normalizar el kernel

#  Función completa
***

In [4]:
def fn_filtro_hanning(directorio_entrada, directorio_salida):
    # Crear el directorio de salida si no existe
    if not os.path.exists(directorio_salida):
        os.makedirs(directorio_salida)

    # Iterar sobre los archivos NetCDF en el directorio de entrada
    for archivo in os.listdir(directorio_entrada):
        if archivo.endswith('.nc'):
            # Cargar el archivo NetCDF
            ruta_archivo = os.path.join(directorio_entrada, archivo)
            ds = xr.open_dataset(ruta_archivo)

            # Crear una matriz para almacenar los perfiles procesados
            num_perfiles = ds.sizes['N_PROF']
            presion_interp = np.arange(0, 6501, 1)
            num_niveles = len(presion_interp)

            def procesar_variable(variable, kernel, presion_interp, num_perfiles):
                perfiles_procesados = np.full((num_perfiles, len(presion_interp)), np.nan)
                for perfil_num in range(num_perfiles):
                    datos = variable.values[perfil_num, :]
                    presion = ds.pressure.values[perfil_num, :]

                    # Eliminar NaN antes de la interpolación
                    valid_indices = ~np.isnan(datos) & ~np.isnan(presion)
                    if not np.any(valid_indices):  # Si no hay datos válidos
                        perfiles_procesados[perfil_num, :] = np.nan
                        continue

                    datos_validos = datos[valid_indices]
                    presion_valida = presion[valid_indices]

                    # Interpolar los valores al eje de presión fijo
                    interp_func = interp1d(presion_valida, datos_validos, bounds_error=False, fill_value=np.nan)
                    datos_interp = interp_func(presion_interp)

                    # Guardar el perfil interpolado
                    perfiles_procesados[perfil_num, :] = datos_interp

                # Calcular la media para cada nivel
                media_por_nivel = np.nanmean(perfiles_procesados, axis=0)

                # Restar la media para cada nivel
                perfiles_sin_media = perfiles_procesados - media_por_nivel

                # Aplicar el filtro Hanning mediante convolución
                perfiles_filtrados = np.array([
                    np.convolve(kernel, perfil, mode='same') for perfil in perfiles_sin_media
                ])

                # Volver a sumar la media en cada nivel
                perfiles_finales = perfiles_filtrados + media_por_nivel
                return perfiles_finales

            # Procesar temperatura
            if 'ctd_temperature' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature
            elif 'ctd_temperature_68' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature_68
            elif 'ctd_temperature_unk' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature_unk
            else:
                variable_temperatura = None

            if variable_temperatura is not None:
                perfiles_finales_temperatura = procesar_variable(variable_temperatura, kernel, presion_interp, num_perfiles)
                ds['ctd_temperature_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_temperatura)

            # Procesar oxígeno
            if 'ctd_oxygen_ml_l' in ds.data_vars:
                variable_oxigeno = ds.ctd_oxygen_ml_l
            elif 'ctd_oxygen' in ds.data_vars:
                variable_oxigeno = ds.ctd_oxygen
            else:
                variable_oxigeno = None

            if variable_oxigeno is not None:
                perfiles_finales_oxigeno = procesar_variable(variable_oxigeno, kernel, presion_interp, num_perfiles)
                ds['ctd_oxygen_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_oxigeno)

            # Procesar salinidad
            if 'ctd_salinity' in ds.data_vars:
                perfiles_finales_salinidad = procesar_variable(ds.ctd_salinity, kernel, presion_interp, num_perfiles)
                ds['ctd_salinity_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_salinidad)

            # Eliminar las variables originales no filtradas
            variables_a_eliminar = ['ctd_temperature', 'ctd_temperature_68', 'ctd_temperature_unk',
                                    'ctd_oxygen_ml_l', 'ctd_oxygen', 'ctd_salinity', 'pressure_qc',
                                    'ctd_oxygen_qc', 'ctd_oxygen_ml_l_qc', 'ctd_temperature_qc',
                                    'ctd_salinity_qc', 'ctd_temperature_unk_qc', 'ctd_temperature_68_qc',
                                    'profile_type', 'geometry_container', 'btm_depth', 'btm_depth_qc']
            for var in variables_a_eliminar:
                if var in ds.data_vars:
                    ds = ds.drop_vars(var)

            # Eliminar coordenadas no deseadas
            coordenadas_a_eliminar = ['sample', 'pressure']
            for coord in coordenadas_a_eliminar:
                if coord in ds.coords:
                    ds = ds.drop_vars(coord)

            # Guardar las nuevas variables al Dataset
            ds['pressure_interp'] = ('pressure_interp', presion_interp)

            # Guardar el archivo procesado en el directorio de salida
            ruta_salida = os.path.join(directorio_salida, archivo)
            ds.to_netcdf(ruta_salida)

            # Cerrar el Dataset
            ds.close()

In [5]:
fn_filtro_hanning(directorio_entrada, directorio_salida)


/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)


In [6]:
# Obtener todas las carpetas en el directorio '../Data/'
directorio_base_entrada = './Data/corrected_sections/'
directorio_base_salida = './Data/corrected_sections_filtrado/'

for carpeta in os.listdir(directorio_base_entrada):
    ruta_carpeta_entrada = os.path.join(directorio_base_entrada, carpeta)
    ruta_carpeta_salida = os.path.join(directorio_base_salida, carpeta)

    # Verificar si es un directorio
    if os.path.isdir(ruta_carpeta_entrada):
        fn_filtro_hanning(ruta_carpeta_entrada, ruta_carpeta_salida)

/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean of empty slice
  media_por_nivel = np.nanmean(perfiles_procesados, axis=0)
/var/folders/qd/9gm8pvgj6m940047172pfyg80000gn/T/ipykernel_23495/3269986578.py:41: RuntimeWarning: Mean o